## 05. Qualitative Validation: The Voice of the Customer

In the previous notebooks, our Random Forest models showed that Digital Friction is the primary predictor of Business Class dissatisfaction. 

To explore how this friction manifests, I am deploying a **Semantic Search (RAG) Pipeline** using OpenAI Embeddings and a Vector Database (Pinecone). By scraping and analyzing real-world discussions from `r/delta`, `r/AmericanAirlines`, and `r/travel`, I aim to identify the specific  modes of failure that drive high-value churn.

### Loading Libraries

In [2]:
import os
from dotenv import load_dotenv
import praw
from openai import OpenAI
import pandas as pd
import time
from pinecone import Pinecone

### Connect APIs

In [3]:
### Setup
load_dotenv('../.env.local')
REDDIT_CLIENT_ID = os.environ.get('REDDIT_CLIENT_ID')
REDDIT_SECRET_ID = os.environ.get('REDDIT_SECRET_ID')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')


In [4]:
### Reddit API
reddit = praw.Reddit(
    client_id = REDDIT_CLIENT_ID,
    client_secret = REDDIT_SECRET_ID,
    user_agent = "researcher"
)
# Session options: controversial, gilded, hot, new, rising, top
print(reddit.read_only)

### OpenAI API
client = OpenAI(api_key=OPENAI_API_KEY)

### Pinecone API
pc = Pinecone(api_key=PINECONE_API_KEY)

# Create a new index 
index_name = "flight-reviews"
dimension = 1536  

existing_indexes = [idx.name for idx in pc.list_indexes()]
if index_name not in existing_indexes:
    print(f"Creating new index: {index_name}")
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec={
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"
            }
        }
    )

index = pc.Index(index_name)
print(f"Connected to index: {index_name}")

True
Connected to index: flight-reviews


### Scrape Reddit

In [5]:
### get comments and posts
target_subreddits = reddit.subreddit("delta+united+AmericanAirlines+travel+awardtravel")
search_query = '''
                ("Business Class" OR "First Class" OR "D1" OR "Polaris") 
                AND 
                ("App" OR "Mobile Boarding" OR "Online Check-in" OR "App Crash" OR "System Error" OR "Website Glitch" OR "Boarding Pass won't load")
               '''

title, text, score, url = [], [], [], []
for submission in target_subreddits.search(
    search_query,
    sort="relevance",
    time_filter="year",
    limit=500):

    title.append(submission.title)
    text.append(submission.selftext)
    score.append(submission.score)
    url.append(submission.url)
    print(f'Unique Submissions {len(set(title))}', end='\r', flush=True)
    time.sleep(0.01)

In [6]:
reddit_df = pd.DataFrame({
    'Title': title,
    'Text': text,
    'Score': score,
    'URL': url
})
display(reddit_df)

,Title,Text,Score,URL
0,Downgraded to economy on a paid first class ti...,"I’m 6’5” and a pretty big dude, so I usually b...",3273,https://i.redd.it/vps8xlmkbo7e1.jpeg
1,Frustrated about Delta employees and upgrades ...,Happened today - going to hide the airport pai...,672,https://www.reddit.com/r/delta/comments/1nox4g...
2,Just WOW,Nothing like pulling up to the airport and fin...,1007,https://i.redd.it/zlqfszmix68e1.jpeg
3,"Is the price to ""upgrade"" to Delta One always ...",I rarely buy a first class ticket because they...,0,https://www.reddit.com/r/delta/comments/1j30eq...
4,Delta -- The worst deal in the skies?,\n\nDiamond Medallion for 7 years straight her...,422,https://www.reddit.com/r/delta/comments/1o50fi...
...,...,...,...,...
236,First Class Children’s Meal Breakfast: What ha...,"Reviewed older posts, checking in for newer in...",0,https://www.reddit.com/r/delta/comments/1maktd...
237,seeking advice on PS/D1 upgrades for Delta tic...,"First time flying a Delta ticketed, AF operate...",1,https://www.reddit.com/r/delta/comments/1lmb1j...
238,Upgrade to delta one?,\nIs it possible for my companion ticket user ...,1,https://www.reddit.com/r/delta/comments/1m8k6n...
239,Upgrade offer,"We are flying RT SEA to LHR in September, prem...",0,https://www.reddit.com/r/delta/comments/1jqsxj...


### Vectorize Posts

In [7]:
# Combine title and text then filter out empty/removed Posts
posts = [f'{title[i]} --> {text[i]}' for i in range(len(title))]
print(f"Total posts to embed: {len(posts):,}")

Total posts to embed: 241


##### Batch Embeddings

In [8]:
# Set batch and comment char length
batch_size = 200
max_chars = 200000

all_vectors = []
total_batches = (len(posts) + batch_size - 1) // batch_size
print(f'Processing {total_batches} batches of up to {batch_size} posts')

for batch_num in range(0, len(posts), batch_size):
    batch = posts[batch_num:batch_num + batch_size]
    batch_char = sum(len(c) for c in batch)

    print(f'Batch {(batch_num // batch_size) + 1}/{total_batches}: Embedding {len(batch)} posts {batch_char:,} chars...', end=" ")

    embeddings_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )

    vectors = [
        (f'id_{batch_num}_{j}', embeddings_response.data[j].embedding, {'text': batch[j], 'upvotes': score[batch_num + j], 'url': url[batch_num + j]})
        for j in range(len(batch))
    ]

    all_vectors.extend(vectors)
    print(f'Done ({len(all_vectors):,} vectors total)')

Processing 2 batches of up to 200 posts
Batch 1/2: Embedding 200 posts 263,222 chars... 

Done (200 vectors total)
Batch 2/2: Embedding 41 posts 54,107 chars... Done (241 vectors total)


##### Upsert to Pinecone

In [9]:
upsert_batch_size = 100
for i in range(0, len(all_vectors), upsert_batch_size):
    batch = all_vectors[i:i+upsert_batch_size]
    index.upsert(vectors=batch)
    print(f'Uploading {i + len(batch)}/{len(all_vectors)} vectors')
print(f'Successfully uploaded {len(all_vectors)} vectors to Pinecone')

Uploading 100/241 vectors
Uploading 200/241 vectors
Uploading 241/241 vectors
Successfully uploaded 241 vectors to Pinecone


### Query Relevant Comments

Instead of searching for keywords (which misses context), I am searching for a **Semantic Concept**.

This query is designed to find "Nearest Neighbors" in the vector space—posts that share the same *sentiment of frustration* regarding premium digital failure, even if they use different words (e.g., "glitch," "error," "won't load").

In [10]:
query = "The airline mobile app crashed and I couldn't access my boarding pass for business class."
query_embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
    ).data[0].embedding

results = index.query(vector=query_embedding, top_k=10, include_metadata=True)

In [11]:
# Save results
markdown_content = f"""# Reddit Query Results

## Query
{query}

## Results (Top {len(results['matches'])} matches)

"""
for i, match in enumerate(results['matches'], 1):
    score = match['score']
    text = match['metadata']['text']
    markdown_content += f""" ### [Result {i}]({match['metadata']['url']}) Upvotes: {match['metadata']['upvotes']}
##### (Similarity Score: {score:.3f})

{text}

---

"""

In [12]:

with open('../datasets/reddit/top_comments.md', 'w', encoding='utf-8') as file:
    file.write(markdown_content)

print(f"Results saved to top_comments.md")

Results saved to top_comments.md


### Qualitative Validation Findings

##### Result 5 (Similarity Score: 0.473)
>I used miles to upgrade to business for the first leg of my flight and worked with an AA agent who confirmed the upgrade and seat assignment. As of this morning, my American app reflected my business class seat. <u>This afternoon I found out that my upgrade triggered a system glitch that canceled the second leg of my trip.</u> Customer service reinstated that leg, but now I'm told the upgrade was never valid and should have never been authorized. I’ve been put on the waitlist again...Feeling super frustrated.

##### Result 7 (Similarity Score: 0.461)
>I’m flying Iberia operated by AA (or maybe I’ve got that backward). <u>On neither Iberia’s nor AA’s websites and apps can I see business class seats on the seating map.</u> I’d like to know the cost to upgrade to business class. What is going on here? 

##### Result 9 (Similarity Score: 0.452)
><u>When are they going to fix these constant app issues?!?!</u> App says my flight has been delayed, but doesn't show me the updated time. 
>App shows me I have been rebooked on a new flight, only to show me the same flights I was already on. App shows a seat number in first class, but still says Comf+ and shows Zone 3. Meanwhile, if I save the ticket to Google Wallet, everything is updated and correct.How can Google get right what Delta can't get right on their own app?

##### Result 8 (Similarity Score: 0.455)
>Flying first Class tomorrow - was pretty sure it said we had access to the lounge when booked tickets and <u>couldn’t find it on the app</u> Called customer service this morning and she assured me with our first class tickets we did and to go thru the app to find a lounge at our airport and to just show boarding pass. How reliable is this info?


##### Result 6 (Similarity Score: 0.472)
>My boarding pass only updates SOMETIMES when my seats get upgraded. <u>I get a lot of first class upgrades, but half the time my boarding pass doesn’t update, and I get a red light at the gate, and they have to physically print a new BP.</u> Is there a way to refresh this on the app?